In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [ ]:
from pathlib import Path
ROOT = Path().resolve().parent  # project root

In [4]:
pacer=pd.read_csv(ROOT / "data/raw/pacer_cases.csv")
cases=pd.read_csv(ROOT / "data/raw/cases.csv")

In [56]:
court_name_mapping = {
    # Alabama
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF ALABAMA": "Alabama Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF ALABAMA": "Alabama Southern District Court",
    "Northern District of Alabama (Eastern)": "Alabama Northern District Court",
    "Northern District of Alabama (Southern)": "Alabama Northern District Court",
    "Northern District of Alabama (Northwestern)": "Alabama Northern District Court",
    "Northern District of Alabama (Northeastern)": "Alabama Northern District Court",
    "Northern District of Alabama (Middle)": "Alabama Northern District Court",
    "Northern District of Alabama (Western)": "Alabama Northern District Court",
    "Southern District of Alabama (Mobile)": "Alabama Southern District Court",

    # Arkansas
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF ARKANSAS": "Arkansas Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF ARKANSAS": "Arkansas Western District Court",
    "Eastern District of Arkansas (Batesville)": "Arkansas Eastern District Court",
    "Eastern District of Arkansas (Helena)": "Arkansas Eastern District Court",
    "Eastern District of Arkansas (Jonesboro)": "Arkansas Eastern District Court",
    "Eastern District of Arkansas (Little Rock)": "Arkansas Eastern District Court",
    "Eastern District of Arkansas (Pine Bluff)": "Arkansas Eastern District Court",
    "Western District of Arkansas (El Dorado)": "Arkansas Western District Court",
    "Western District of Arkansas (Fayetteville)": "Arkansas Western District Court",
    "Western District of Arkansas (Fort Smith)": "Arkansas Western District Court",
    "Western District of Arkansas (Harrison)": "Arkansas Western District Court",
    "Western District of Arkansas (Hot Springs)": "Arkansas Western District Court",
    "Western District of Arkansas (Texarkana)": "Arkansas Western District Court",

    # Arizona
    "UNITED STATES DISTRICT COURT DISTRICT OF ARIZONA": "Arizona District Court",
    "DISTRICT OF ARIZONA (Phoenix Division)": "Arizona District Court",
    "DISTRICT OF ARIZONA (Prescott Division)": "Arizona District Court",
    "DISTRICT OF ARIZONA (Tucson Division)": "Arizona District Court",

    # California
    "UNITED STATES DISTRICT COURT CENTRAL DISTRICT OF CALIFORNIA": "California Central District Court",
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF CALIFORNIA": "California Eastern District Court",
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF CALIFORNIA": "California Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF CALIFORNIA": "California Southern District Court",
    "UNITED STATES DISTRICT COURT for the CENTRAL DISTRICT OF CALIFORNIA (Western Division - Los Angeles)": "California Central District Court",
    "UNITED STATES DISTRICT COURT for the CENTRAL DISTRICT OF CALIFORNIA (Eastern Division - Riverside)": "California Central District Court",
    "UNITED STATES DISTRICT COURT for the CENTRAL DISTRICT OF CALIFORNIA (Southern Division - Santa Ana)": "California Central District Court",
    "California Northern District (San Francisco)": "California Northern District Court",
    "California Northern District (Oakland)": "California Northern District Court",
    "California Northern District (San Jose)": "California Northern District Court",
    "Eastern District of California - Live System (Fresno)": "California Eastern District Court",
    "Eastern District of California - Live System (Sacramento)": "California Eastern District Court",
    "Southern District of California (San Diego)": "California Southern District Court",

    # Colorado
    "UNITED STATES DISTRICT COURT DISTRICT OF COLORADO": "Colorado District Court",
    "District of Colorado (Denver)": "Colorado District Court",

    # Connecticut
    "UNITED STATES DISTRICT COURT DISTRICT OF CONNECTICUT": "Connecticut District Court",
    "United States District Court for the District of Connecticut (Hartford)": "Connecticut District Court",
    "United States District Court for the District of Connecticut (New Haven)": "Connecticut District Court",
    "United States District Court for the District of Connecticut (Bridgeport)": "Connecticut District Court",

    # DC
    "UNITED STATES DISTRICT COURT DISTRICT OF COLUMBIA": "District Of Columbia District Court",
    "District of Columbia (Washington, DC)": "District Of Columbia District Court",

    # Delaware
    "UNITED STATES DISTRICT COURT DISTRICT OF DELAWARE": "Delaware District Court",
    "District of Delaware (Wilmington)": "Delaware District Court",

    # Florida
    "UNITED STATES DISTRICT COURT MIDDLE DISTRICT OF FLORIDA": "Florida Middle District Court",
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF FLORIDA": "Florida Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF FLORIDA": "Florida Southern District Court",
    "Middle District of Florida (Ft. Myers)": "Florida Middle District Court",
    "Middle District of Florida (Jacksonville)": "Florida Middle District Court",
    "Middle District of Florida (Ocala)": "Florida Middle District Court",
    "Middle District of Florida (Orlando)": "Florida Middle District Court",
    "Middle District of Florida (Tampa)": "Florida Middle District Court",
    "Northern District of Florida (Gainesville)": "Florida Northern District Court",
    "Northern District of Florida (Panama City)": "Florida Northern District Court",
    "Northern District of Florida (Pensacola)": "Florida Northern District Court",
    "Northern District of Florida (Tallahassee)": "Florida Northern District Court",
    "Southern District of Florida (Ft Lauderdale)": "Florida Southern District Court",
    "Southern District of Florida (Ft Pierce)": "Florida Southern District Court",
    "Southern District of Florida (Key West)": "Florida Southern District Court",
    "Southern District of Florida (Miami)": "Florida Southern District Court",
    "Southern District of Florida (West Palm Beach)": "Florida Southern District Court",

    # Georgia
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF GEORGIA": "Georgia Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF GEORGIA": "Georgia Southern District Court",
    "UNITED STATES DISTRICT COURT MIDDLE DISTRICT OF GEORGIA": "Georgia Middle District Court",
    "Northern District of Georgia (Atlanta)": "Georgia Northern District Court",
    "Northern District of Georgia (Gainesville)": "Georgia Northern District Court",
    "Northern District of Georgia (Newnan)": "Georgia Northern District Court",
    "Northern District of Georgia (Rome)": "Georgia Northern District Court",
    "Southern District of Georgia (Augusta)": "Georgia Southern District Court",
    "Southern District of Georgia (Brunswick)": "Georgia Southern District Court",
    "Southern District of Georgia (Dublin)": "Georgia Southern District Court",
    "Southern District of Georgia (Savannah)": "Georgia Southern District Court",
    "Southern District of Georgia (Statesboro)": "Georgia Southern District Court",
    "Southern District of Georgia (Waycross)": "Georgia Southern District Court",
    "Middle District of Georgia (Albany)": "Georgia Middle District Court",
    "Middle District of Georgia (Athens)": "Georgia Middle District Court",
    "Middle District of Georgia (Columbus)": "Georgia Middle District Court",
    "Middle District of Georgia (Macon)": "Georgia Middle District Court",
    "Middle District of Georgia (Thomasville)": "Georgia Middle District Court",
    "Middle District of Georgia (Valdosta)": "Georgia Middle District Court",

    # Guam
    "District Court of Guam (Hagatna)": "Guam District Court",

    # Hawaii
    "UNITED STATES DISTRICT COURT DISTRICT OF HAWAII": "Hawaii District Court",
    "District of Hawaii (Hawaii)": "Hawaii District Court",

    # Idaho
    "UNITED STATES DISTRICT COURT DISTRICT OF IDAHO": "Idaho District Court",
    "District of Idaho (LIVE Database)Version 6.1 (Boise - Southern)": "Idaho District Court",
    "District of Idaho (LIVE Database)Version 6.1 (CDA - Northern)": "Idaho District Court",
    "District of Idaho (LIVE Database)Version 6.1 (Moscow - Central)": "Idaho District Court",
    "District of Idaho (LIVE Database)Version 6.1 (Pocatello - Eastern)": "Idaho District Court",

    # Illinois
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF ILLINOIS": "Illinois Northern District Court",
    "UNITED STATES DISTRICT COURT CENTRAL DISTRICT OF ILLINOIS": "Illinois Central District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF ILLINOIS": "Illinois Southern District Court",
    "Northern District of Illinois - CM/ECF LIVE, Ver 6,1 (Chicago)": "Illinois Northern District Court",
    "Northern District of Illinois - CM/ECF LIVE, Ver 6,1 (Rockford)": "Illinois Northern District Court",
    "CENTRAL DISTRICT OF ILLINOIS (Peoria)": "Illinois Central District Court",
    "CENTRAL DISTRICT OF ILLINOIS (Rock Island)": "Illinois Central District Court",
    "CENTRAL DISTRICT OF ILLINOIS (Springfield)": "Illinois Central District Court",
    "CENTRAL DISTRICT OF ILLINOIS (Urbana)": "Illinois Central District Court",
    "Southern District of Illinois (Benton)": "Illinois Southern District Court",
    "Southern District of Illinois (East St. Louis)": "Illinois Southern District Court",

    # Indiana
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF INDIANA": "Indiana Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF INDIANA": "Indiana Southern District Court",
    "USDC Northern Indiana (Fort Wayne)": "Indiana Northern District Court",
    "USDC Northern Indiana (Hammond)": "Indiana Northern District Court",
    "USDC Northern Indiana (Lafayette)": "Indiana Northern District Court",
    "USDC Northern Indiana (South Bend)": "Indiana Northern District Court",
    "Southern District of Indiana (Evansville)": "Indiana Southern District Court",
    "Southern District of Indiana (Indianapolis)": "Indiana Southern District Court",
    "Southern District of Indiana (New Albany)": "Indiana Southern District Court",

    # Iowa
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF IOWA": "Iowa Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF IOWA": "Iowa Southern District Court",
    "Northern District of Iowa (Cedar Rapids)": "Iowa Northern District Court",
    "Northern District of Iowa (Central Division)": "Iowa Northern District Court",
    "Northern District of Iowa (Eastern Dubuque)": "Iowa Northern District Court",
    "Northern District of Iowa (Eastern Waterloo)": "Iowa Northern District Court",
    "Northern District of Iowa (Western Division)": "Iowa Northern District Court",
    "Southern District of Iowa (Central)": "Iowa Southern District Court",
    "Southern District of Iowa (Davenport)": "Iowa Southern District Court",
    "Southern District of Iowa (Western)": "Iowa Southern District Court",

    # Kansas
    "UNITED STATES DISTRICT COURT DISTRICT OF KANSAS": "Kansas District Court",
    "DISTRICT OF KANSAS (Kansas City)": "Kansas District Court",
    "DISTRICT OF KANSAS (Topeka)": "Kansas District Court",
    "DISTRICT OF KANSAS (Wichita)": "Kansas District Court",

    # Kentucky
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF KENTUCKY": "Kentucky Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF KENTUCKY": "Kentucky Western District Court",
    "Eastern District of Kentucky (Ashland)": "Kentucky Eastern District Court",
    "Eastern District of Kentucky (Covington)": "Kentucky Eastern District Court",
    "Eastern District of Kentucky (Frankfort)": "Kentucky Eastern District Court",
    "Eastern District of Kentucky (Lexington)": "Kentucky Eastern District Court",
    "Eastern District of Kentucky (London)": "Kentucky Eastern District Court",
    "Eastern District of Kentucky (Pikeville)": "Kentucky Eastern District Court",
    "Western District of Kentucky (Bowling Green)": "Kentucky Western District Court",
    "Western District of Kentucky (Louisville)": "Kentucky Western District Court",
    "Western District of Kentucky (Owensboro)": "Kentucky Western District Court",
    "Western District of Kentucky (Paducah)": "Kentucky Western District Court",

    # Louisiana
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF LOUISIANA": "Louisiana Eastern District Court",
    "UNITED STATES DISTRICT COURT MIDDLE DISTRICT OF LOUISIANA": "Louisiana Middle District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF LOUISIANA": "Louisiana Western District Court",
    "Eastern District of Louisiana (New Orleans)": "Louisiana Eastern District Court",
    "Middle District of Louisiana (Baton Rouge)": "Louisiana Middle District Court",
    "Western District of Louisiana (Alexandria)": "Louisiana Western District Court",
    "Western District of Louisiana (Lafayette)": "Louisiana Western District Court",
    "Western District of Louisiana (Lake Charles)": "Louisiana Western District Court",
    "Western District of Louisiana (Monroe)": "Louisiana Western District Court",
    "Western District of Louisiana (Opelousas)": "Louisiana Western District Court",
    "Western District of Louisiana (Shreveport)": "Louisiana Western District Court",

    # Maine
    "UNITED STATES DISTRICT COURT DISTRICT OF MAINE": "Maine District Court",
    "District of Maine (Bangor)": "Maine District Court",
    "District of Maine (Portland)": "Maine District Court",

    # Maryland
    "UNITED STATES DISTRICT COURT DISTRICT OF MARYLAND": "Maryland District Court",
    "District of Maryland (Baltimore)": "Maryland District Court",
    "District of Maryland (Greenbelt)": "Maryland District Court",
    "District of Maryland (2)": "Maryland District Court",

    # Massachusetts
    "UNITED STATES DISTRICT COURT DISTRICT OF MASSACHUSETTS": "Massachusetts District Court",
    "District of Massachusetts (Boston)": "Massachusetts District Court",
    "District of Massachusetts (Springfield)": "Massachusetts District Court",
    "District of Massachusetts (Worcester)": "Massachusetts District Court",

    # Michigan
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF MICHIGAN": "Michigan Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF MICHIGAN": "Michigan Western District Court",
    "Eastern District of Michigan (Ann Arbor)": "Michigan Eastern District Court",
    "Eastern District of Michigan (Bay City)": "Michigan Eastern District Court",
    "Eastern District of Michigan (Detroit)": "Michigan Eastern District Court",
    "Eastern District of Michigan (Flint)": "Michigan Eastern District Court",
    "Eastern District of Michigan (Port Huron)": "Michigan Eastern District Court",
    "Western District of Michigan (Northern Division (2))": "Michigan Western District Court",
    "Western District of Michigan (Southern Division (1))": "Michigan Western District Court",
    "Western District of Michigan (Southern Division (4))": "Michigan Western District Court",
    "Western District of Michigan (Southern Division (5))": "Michigan Western District Court",

    # Minnesota
    "UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA": "Minnesota District Court",
    "U.S. District of Minnesota (DMN)": "Minnesota District Court",
    "U.S. District of Minnesota (DUL)": "Minnesota District Court",
    "U.S. District of Minnesota (FF)": "Minnesota District Court",
    "U.S. District of Minnesota (MPLS)": "Minnesota District Court",
    "U.S. District of Minnesota (STP)": "Minnesota District Court",

    # Mississippi
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF MISSISSIPPI": "Mississippi Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF MISSISSIPPI": "Mississippi Southern District Court",
    "Northern District of Mississippi (Aberdeen Division)": "Mississippi Northern District Court",
    "Northern District of Mississippi (Delta Division)": "Mississippi Northern District Court",
    "Northern District of Mississippi (Greenville Division)": "Mississippi Northern District Court",
    "Northern District of Mississippi (Oxford Division)": "Mississippi Northern District Court",
    "Southern District of Mississippi (Eastern (Hattiesburg))": "Mississippi Southern District Court",
    "Southern District of Mississippi (Eastern)": "Mississippi Southern District Court",
    "Southern District of Mississippi (Northern (Jackson))": "Mississippi Southern District Court",
    "Southern District of Mississippi (Southern)": "Mississippi Southern District Court",
    "Southern District of Mississippi (Western)": "Mississippi Southern District Court",

    # Missouri
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF MISSOURI": "Missouri Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF MISSOURI": "Missouri Western District Court",
    "Eastern District of Missouri (Cape Girardeau)": "Missouri Eastern District Court",
    "Eastern District of Missouri (Hannibal)": "Missouri Eastern District Court",
    "Eastern District of Missouri (St. Louis)": "Missouri Eastern District Court",
    "Western District of Missouri (Jefferson City)": "Missouri Western District Court",
    "Western District of Missouri (Joplin)": "Missouri Western District Court",
    "Western District of Missouri (Kansas City)": "Missouri Western District Court",
    "Western District of Missouri (Springfield)": "Missouri Western District Court",
    "Western District of Missouri (St. Joseph)": "Missouri Western District Court",

    # Montana
    "UNITED STATES DISTRICT COURT DISTRICT OF MONTANA": "Montana District Court",
    "District Of Montana (Billings)": "Montana District Court",
    "District Of Montana (Butte)": "Montana District Court",
    "District Of Montana (Great Falls)": "Montana District Court",
    "District Of Montana (Helena)": "Montana District Court",
    "District Of Montana (Missoula)": "Montana District Court",

    # Nebraska
    "UNITED STATES DISTRICT COURT DISTRICT OF NEBRASKA": "Nebraska District Court",
    "District of Nebraska (4 Lincoln)": "Nebraska District Court",
    "District of Nebraska (7 North Platte)": "Nebraska District Court",
    "District of Nebraska (8 Omaha)": "Nebraska District Court",

    # Nevada
    "UNITED STATES DISTRICT COURT DISTRICT OF NEVADA": "Nevada District Court",
    "District of Nevada (Las Vegas)": "Nevada District Court",
    "District of Nevada (Reno)": "Nevada District Court",

    # New Hampshire
    "UNITED STATES DISTRICT COURT DISTRICT OF NEW HAMPSHIRE": "New Hampshire District Court",
    "District of New Hampshire (Concord)": "New Hampshire District Court",

    # New Jersey
    "UNITED STATES DISTRICT COURT DISTRICT OF NEW JERSEY": "New Jersey District Court",
    "District of New Jersey [LIVE] (Camden)": "New Jersey District Court",
    "District of New Jersey [LIVE] (Newark)": "New Jersey District Court",
    "District of New Jersey [LIVE] (Trenton)": "New Jersey District Court",

    # New Mexico
    "UNITED STATES DISTRICT COURT DISTRICT OF NEW MEXICO": "New Mexico District Court",
    "District of New Mexico - Version 6.1 (Albuquerque)": "New Mexico District Court",
    "District of New Mexico - Version 6.1 (Las Cruces)": "New Mexico District Court",
    "District of New Mexico - Version 6.1 (Santa Fe)": "New Mexico District Court",

    # New York
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF NEW YORK": "New York Eastern District Court",
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF NEW YORK": "New York Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF NEW YORK": "New York Southern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF NEW YORK": "New York Western District Court",
    "Eastern District of New York (Brooklyn)": "New York Eastern District Court",
    "Eastern District of New York (Central Islip)": "New York Eastern District Court",
    "Eastern District of New York (Hauppauge)": "New York Eastern District Court",
    "Eastern District of New York (Uniondale)": "New York Eastern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1] (Albany)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1] (Binghamton)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1] (Syracuse)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1.1] (Albany)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1.1] (Binghamton)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1.1] (Plattsburgh)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1.1] (Syracuse)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1.1] (Utica)": "New York Northern District Court",
    "Northern District of New York - Main Office (Syracuse) [LIVE - Version 6.1.1] (Watertown)": "New York Northern District Court",
    "Southern District of New York (Foley Square)": "New York Southern District Court",
    "Southern District of New York (Foley Square - Suspense)": "New York Southern District Court",
    "Southern District of New York (White Plains)": "New York Southern District Court",
    "U.S. District Court, Western District of New York (Buffalo)": "New York Western District Court",
    "U.S. District Court, Western District of New York (Rochester)": "New York Western District Court",

    # North Carolina
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF NORTH CAROLINA": "North Carolina Eastern District Court",
    "UNITED STATES DISTRICT COURT MIDDLE DISTRICT OF NORTH CAROLINA": "North Carolina Middle District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF NORTH CAROLINA": "North Carolina Western District Court",
    "EASTERN DISTRICT OF NORTH CAROLINA (Eastern Division)": "North Carolina Eastern District Court",
    "EASTERN DISTRICT OF NORTH CAROLINA (Northern Division)": "North Carolina Eastern District Court",
    "EASTERN DISTRICT OF NORTH CAROLINA (Old fayetteville Division)": "North Carolina Eastern District Court",
    "EASTERN DISTRICT OF NORTH CAROLINA (Southern Division)": "North Carolina Eastern District Court",
    "EASTERN DISTRICT OF NORTH CAROLINA (Western Division)": "North Carolina Eastern District Court",
    "North Carolina Middle District (NCMD)": "North Carolina Middle District Court",
    "North Carolina Middle District (Greensboro)": "North Carolina Middle District Court",
    "North Carolina Middle District (Rockingham)": "North Carolina Middle District Court",
    "North Carolina Middle District (Salisbury)": "North Carolina Middle District Court",
    "North Carolina Middle District (Winston-Salem)": "North Carolina Middle District Court",
    "Western District of North Carolina (Asheville)": "North Carolina Western District Court",
    "Western District of North Carolina (Charlotte)": "North Carolina Western District Court",
    "Western District of North Carolina (Shelby)": "North Carolina Western District Court",
    "Western District of North Carolina (Statesville)": "North Carolina Western District Court",

    # North Dakota
    "UNITED STATES DISTRICT COURT DISTRICT OF NORTH DAKOTA": "North Dakota District Court",
    "District of North Dakota (2)": "North Dakota District Court",
    "District of North Dakota (4)": "North Dakota District Court",
    "District of North Dakota (Eastern)": "North Dakota District Court",
    "District of North Dakota (Western)": "North Dakota District Court",

    # Ohio
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF OHIO": "Ohio Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF OHIO": "Ohio Southern District Court",
    "Northern District of Ohio (Akron)": "Ohio Northern District Court",
    "Northern District of Ohio (Cleveland)": "Ohio Northern District Court",
    "Northern District of Ohio (Toledo)": "Ohio Northern District Court",
    "Northern District of Ohio (Youngstown)": "Ohio Northern District Court",
    "Southern District of Ohio (Cincinnati)": "Ohio Southern District Court",
    "Southern District of Ohio (Columbus)": "Ohio Southern District Court",
    "Southern District of Ohio (Dayton)": "Ohio Southern District Court",

    # Oklahoma
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF OKLAHOMA": "Oklahoma Northern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF OKLAHOMA": "Oklahoma Western District Court",
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF OKLAHOMA": "Oklahoma Eastern District Court",
    "U.S. District Court for the Northern District of Oklahoma (Tulsa)": "Oklahoma Northern District Court",
    "Western District of Oklahoma[LIVE] (Oklahoma City)": "Oklahoma Western District Court",
    "Eastern District of Oklahoma (Muskogee)": "Oklahoma Eastern District Court",

    # Oregon
    "UNITED STATES DISTRICT COURT DISTRICT OF OREGON": "Oregon District Court",
    "District of Oregon (Eugene (6))": "Oregon District Court",
    "District of Oregon (Medford (1))": "Oregon District Court",
    "District of Oregon (Pendleton (2))": "Oregon District Court",
    "District of Oregon (Portland (3))": "Oregon District Court",

    # Pennsylvania
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF PENNSYLVANIA": "Pennsylvania Eastern District Court",
    "UNITED STATES DISTRICT COURT MIDDLE DISTRICT OF PENNSYLVANIA": "Pennsylvania Middle District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF PENNSYLVANIA": "Pennsylvania Western District Court",
    "Eastern District of Pennsylvania (Allentown)": "Pennsylvania Eastern District Court",
    "Eastern District of Pennsylvania (Philadelphia)": "Pennsylvania Eastern District Court",
    "Middle District of Pennsylvania (Harrisburg)": "Pennsylvania Middle District Court",
    "Middle District of Pennsylvania (Scranton)": "Pennsylvania Middle District Court",
    "Middle District of Pennsylvania (Williamsport)": "Pennsylvania Middle District Court",
    "Western District of Pennsylvania (Erie)": "Pennsylvania Western District Court",
    "Western District of Pennsylvania (Johnstown)": "Pennsylvania Western District Court",
    "Western District of Pennsylvania (Pittsburgh)": "Pennsylvania Western District Court",

    # Puerto Rico
    "UNITED STATES DISTRICT COURT DISTRICT OF PUERTO RICO": "Puerto Rico District Court",
    "District of Puerto Rico (San Juan)": "Puerto Rico District Court",

    # Rhode Island
    "UNITED STATES DISTRICT COURT DISTRICT OF RHODE ISLAND": "Rhode Island District Court",
    "District of Rhode Island (Providence)": "Rhode Island District Court",

    # South Carolina
    "UNITED STATES DISTRICT COURT DISTRICT OF SOUTH CAROLINA": "South Carolina District Court",
    "District of South Carolina (Aiken)": "South Carolina District Court",
    "District of South Carolina (Anderson/Greenwood)": "South Carolina District Court",
    "District of South Carolina (Beaufort)": "South Carolina District Court",
    "District of South Carolina (Charleston)": "South Carolina District Court",
    "District of South Carolina (Columbia)": "South Carolina District Court",
    "District of South Carolina (Florence)": "South Carolina District Court",
    "District of South Carolina (Greenville)": "South Carolina District Court",
    "District of South Carolina (Orangeburg)": "South Carolina District Court",
    "District of South Carolina (Rock Hill)": "South Carolina District Court",
    "District of South Carolina (Spartanburg)": "South Carolina District Court",

    # South Dakota
    "UNITED STATES DISTRICT COURT DISTRICT OF SOUTH DAKOTA": "South Dakota District Court",
    "District of South Dakota (Central Division)": "South Dakota District Court",
    "District of South Dakota (Northern Division)": "South Dakota District Court",
    "District of South Dakota (Southern Division)": "South Dakota District Court",
    "District of South Dakota (Western Division)": "South Dakota District Court",

    # Tennessee
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF TENNESSEE": "Tennessee Eastern District Court",
    "UNITED STATES DISTRICT COURT MIDDLE DISTRICT OF TENNESSEE": "Tennessee Middle District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF TENNESSEE": "Tennessee Western District Court",
    "U.S. District Court - Eastern District of Tennessee (Chattanooga)": "Tennessee Eastern District Court",
    "U.S. District Court - Eastern District of Tennessee (Greeneville)": "Tennessee Eastern District Court",
    "U.S. District Court - Eastern District of Tennessee (Knoxville)": "Tennessee Eastern District Court",
    "U.S. District Court - Eastern District of Tennessee (Winchester)": "Tennessee Eastern District Court",
    "Middle District of Tennessee (Columbia)": "Tennessee Middle District Court",
    "Middle District of Tennessee (Cookeville)": "Tennessee Middle District Court",
    "Middle District of Tennessee (Nashville)": "Tennessee Middle District Court",
    "Western District of Tennessee (Jackson)": "Tennessee Western District Court",
    "Western District of Tennessee (Memphis)": "Tennessee Western District Court",

    # Texas
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF TEXAS": "Texas Eastern District Court",
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF TEXAS": "Texas Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF TEXAS": "Texas Southern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF TEXAS": "Texas Western District Court",
    "Eastern District of TEXAS (Beaumont)": "Texas Eastern District Court",
    "Eastern District of TEXAS (Lufkin)": "Texas Eastern District Court",
    "Eastern District of TEXAS (Marshall)": "Texas Eastern District Court",
    "Eastern District of TEXAS (Paris)": "Texas Eastern District Court",
    "Eastern District of TEXAS (Sherman)": "Texas Eastern District Court",
    "Eastern District of TEXAS (Texarkana)": "Texas Eastern District Court",
    "Eastern District of TEXAS (Tyler)": "Texas Eastern District Court",
    "Northern District of Texas (Abilene)": "Texas Northern District Court",
    "Northern District of Texas (Amarillo)": "Texas Northern District Court",
    "Northern District of Texas (Dallas)": "Texas Northern District Court",
    "Northern District of Texas (Fort Worth)": "Texas Northern District Court",
    "Northern District of Texas (Lubbock)": "Texas Northern District Court",
    "Northern District of Texas (San Angelo)": "Texas Northern District Court",
    "Northern District of Texas (Wichita Falls)": "Texas Northern District Court",
    "SOUTHERN DISTRICT OF TEXAS (Brownsville)": "Texas Southern District Court",
    "SOUTHERN DISTRICT OF TEXAS (Corpus Christi)": "Texas Southern District Court",
    "SOUTHERN DISTRICT OF TEXAS (Galveston)": "Texas Southern District Court",
    "SOUTHERN DISTRICT OF TEXAS (Houston)": "Texas Southern District Court",
    "SOUTHERN DISTRICT OF TEXAS (Laredo)": "Texas Southern District Court",
    "SOUTHERN DISTRICT OF TEXAS (McAllen)": "Texas Southern District Court",
    "SOUTHERN DISTRICT OF TEXAS (Victoria)": "Texas Southern District Court",
    "Western District of Texas (Austin)": "Texas Western District Court",
    "Western District of Texas (Del Rio)": "Texas Western District Court",
    "Western District of Texas (El Paso)": "Texas Western District Court",
    "Western District of Texas (Midland)": "Texas Western District Court",
    "Western District of Texas (San Antonio)": "Texas Western District Court",
    "Western District of Texas (Waco)": "Texas Western District Court",
    "U.S. District Court (7)": "Texas Western District Court",


    # Utah
    "UNITED STATES DISTRICT COURT DISTRICT OF UTAH": "Utah District Court",
    "District of Utah (Central)": "Utah District Court",
    "District of Utah (Northern)": "Utah District Court",

    # Vermont
    "UNITED STATES DISTRICT COURT DISTRICT OF VERMONT": "Vermont District Court",
    "District of Vermont (Brattleboro)": "Vermont District Court",
    "District of Vermont (Burlington)": "Vermont District Court",
    "District of Vermont (Rutland)": "Vermont District Court",

    # Virgin Islands
    "UNITED STATES DISTRICT COURT DISTRICT COURT OF THE VIRGIN ISLANDS": "Virgin Islands",

    # Virginia
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF VIRGINIA": "Virginia Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF VIRGINIA": "Virginia Western District Court",
    "Eastern District of Virginia - (Alexandria)": "Virginia Eastern District Court",
    "Eastern District of Virginia - (Newport News)": "Virginia Eastern District Court",
    "Eastern District of Virginia - (Norfolk)": "Virginia Eastern District Court",
    "Eastern District of Virginia - (Richmond)": "Virginia Eastern District Court",
    "Western District of Virginia (Abingdon)": "Virginia Western District Court",
    "Western District of Virginia (Big Stone Gap)": "Virginia Western District Court",
    "Western District of Virginia (Charlottesville)": "Virginia Western District Court",
    "Western District of Virginia (Danville)": "Virginia Western District Court",
    "Western District of Virginia (Harrisonburg)": "Virginia Western District Court",
    "Western District of Virginia (Lynchburg)": "Virginia Western District Court",
    "Western District of Virginia (Roanoke)": "Virginia Western District Court",

    # Washington
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF WASHINGTON": "Washington Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF WASHINGTON": "Washington Western District Court",
    "U.S. District Court (Spokane)": "Washington Eastern District Court",
    "United States District Court for the Western District of Washington (Seattle)": "Washington Western District Court",
    "United States District Court for the Western District of Washington (Tacoma)": "Washington Western District Court",

    # West Virginia
    "UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF WEST VIRGINIA": "West Virginia Northern District Court",
    "UNITED STATES DISTRICT COURT SOUTHERN DISTRICT OF WEST VIRGINIA": "West Virginia Southern District Court",
    "Northern District of West Virginia (Clarksburg)": "West Virginia Northern District Court",
    "Northern District of West Virginia (Martinsburg)": "West Virginia Northern District Court",
    "Northern District of West Virginia (Wheeling)": "West Virginia Northern District Court",
    "Southern District of West Virginia (Beckley)": "West Virginia Southern District Court",
    "Southern District of West Virginia (Bluefield)": "West Virginia Southern District Court",
    "Southern District of West Virginia (Charleston)": "West Virginia Southern District Court",
    "Southern District of West Virginia (Huntington)": "West Virginia Southern District Court",
    "Southern District of West Virginia (Parkersburg)": "West Virginia Southern District Court",

    # Wisconsin
    "UNITED STATES DISTRICT COURT EASTERN DISTRICT OF WISCONSIN": "Wisconsin Eastern District Court",
    "UNITED STATES DISTRICT COURT WESTERN DISTRICT OF WISCONSIN": "Wisconsin Western District Court",
    "Eastern District of Wisconsin (Green Bay)": "Wisconsin Eastern District Court",
    "Eastern District of Wisconsin (Milwaukee)": "Wisconsin Eastern District Court",
    "Western District of Wisconsin (Madison)": "Wisconsin Western District Court",

    # Wyoming
    "UNITED STATES DISTRICT COURT DISTRICT OF WYOMING": "Wyoming District Court",
    "District of Wyoming (Casper)": "Wyoming District Court",
    "District of Wyoming (Cheyenne)": "Wyoming District Court",
  
    
}

# Apply to cases df
cases['court_name_normalized'] = cases['court_name'].map(court_name_mapping).fillna(cases['court_name'])

# Verify coverage
unmapped = cases[~cases['court_name'].isin(court_name_mapping)]['court_name'].unique()
print(f"Unmapped court names: {len(unmapped)}")
print(unmapped)

Unmapped court names: 0
<StringArray>
[]
Length: 0, dtype: str


In [57]:
cases['court_name_normalized'].value_counts()

court_name_normalized
Texas Eastern District Court          9744
Delaware District Court               6600
California Central District Court     6405
California Northern District Court    4268
Illinois Northern District Court      3805
                                      ... 
Wyoming District Court                  33
Alabama Southern District Court         31
Oklahoma Eastern District Court         10
Guam District Court                      2
Virgin Islands                           1
Name: count, Length: 91, dtype: int64

In [90]:
pacer = pacer[~pacer['case_number'].str.contains('font', case=False, na=False)]
print(f"Pacer: {len(pacer)} -> {len(pacer_clean)}")

Pacer: 74852 -> 74853


In [91]:
def normalize_case_number(case_num):
    if pd.isna(case_num):
        return case_num

    case_num = case_num.strip()
    case_num = case_num.replace('CIVIL DOCKET FOR CASE #', '').strip()

    match = re.match(r'(\d+):(\d{2})-(\w+)-(\d+)(-\w+)?', case_num)

    if match:
        prefix, year, case_type, number, suffix = match.groups()

        year = int(year)
        year = 1900 + year if year >= 50 else 2000 + year

        case_type = case_type.lower()

        return f"{prefix}:{year}-{case_type}-{number}"

    return case_num  # keep original if it fails

In [92]:
cases['case_number_norm'] = cases['case_number'].apply(normalize_case_number)
pacer['p_case_number_norm'] = pacer['case_number'].apply(normalize_case_number)

In [105]:
cases[cases.duplicated(subset=['case_number_normalized', 'court_name_normalized'], keep=False)].sort_values(['case_number_normalized', 'court_name_normalized'])

,case_row_id,case_number,pacer_id,case_name,court_name,assigned_to,referred_to,case_cause,jurisdictional_basis,demand,...,related_case,settlement,date_filed,date_closed,date_last_filed,case_number_normalized,is_matched,identifier,court_name_normalized,case_number_norm
6382,12558,1:05-cv-00590,46693.0,"Jackson Products, Inc. v. Fibre-Metal Products...",UNITED STATES DISTRICT COURT WESTERN DISTRICT ...,Judge Robert J. Jonker,NaN,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2005-09-01,2009-03-24,NaN,1:2005-cv-00590,True,1:2005-cv-00590_Michigan Western District Court,Michigan Western District Court,1:2005-cv-00590
6383,13241,1:05-cv-00590,48267.0,Balder Optoelectronic Elements and Measuring S...,UNITED STATES DISTRICT COURT WESTERN DISTRICT ...,Judge Robert J. Jonker,NaN,28:2201 Declaratory Judgement,Federal Question,NaN,...,NaN,NaN,2006-03-15,2009-03-24,NaN,1:2005-cv-00590,True,1:2005-cv-00590_Michigan Western District Court,Michigan Western District Court,1:2005-cv-00590
7195,13373,1:06-cv-00546,37045.0,"Ronald A. Katz Technology Licensing, L.P. v. T...",UNITED STATES DISTRICT COURT DISTRICT OF DELAWARE,Judge Gregory M. Sleet,NaN,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2006-09-01,2007-04-11,2007-11-06,1:2006-cv-00546,True,1:2006-cv-00546_Delaware District Court,Delaware District Court,1:2006-cv-00546
7196,13371,1:06-cv-00546,37036.0,Ronald A Katz Technology Licensing LP v. TD Ba...,UNITED STATES DISTRICT COURT DISTRICT OF DELAWARE,Judge Gregory M. Sleet,NaN,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2006-09-01,2007-04-11,2007-08-29,1:2006-cv-00546,True,1:2006-cv-00546_Delaware District Court,Delaware District Court,1:2006-cv-00546
14929,21111,1:13-cv-00009,50659.0,InterDigital Communications Inc. et al v. Huaw...,UNITED STATES DISTRICT COURT DISTRICT OF DELAWARE,Judge Richard G. Andrews,NaN,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2013-01-02,2013-12-30,NaN,1:2013-cv-00009,True,1:2013-cv-00009_Delaware District Court,Delaware District Court,1:2013-cv-00009
14930,21113,1:13-cv-00009,50660.0,InterDigital Communications Inc. et al v. ZTE ...,UNITED STATES DISTRICT COURT DISTRICT OF DELAWARE,Judge Richard G. Andrews,NaN,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2013-01-02,NaN,2015-04-07,1:2013-cv-00009,True,1:2013-cv-00009_Delaware District Court,Delaware District Court,1:2013-cv-00009
39433,45342,2:14-cv-00153,92166.0,In Re: BRCA1- and BRCA2- Based Hereditary Canc...,UNITED STATES DISTRICT COURT DISTRICT OF UTAH,Judge Robert J. Shelby,NaN,35:0271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2014-02-25,2015-03-09,NaN,2:2014-cv-00153,True,2:2014-cv-00153_Utah District Court,Utah District Court,2:2014-cv-00153
39434,43887,2:14-cv-00153,92191.0,"Invitae Corporation v. Myriad Genetics, Inc",UNITED STATES DISTRICT COURT DISTRICT OF UTAH,Judge Robert J. Shelby,NaN,15:1126 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2014-03-04,2015-02-10,NaN,2:2014-cv-00153,True,2:2014-cv-00153_Utah District Court,Utah District Court,2:2014-cv-00153
50100,53489,3:05-cv-04374,34369.0,"Ultra Clean Technology Systems & Service, Inc....",UNITED STATES DISTRICT COURT NORTHERN DISTRICT...,Hon. Maxine M. Chesney,Magistrate Judge James Larson,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2005-09-02,2007-11-30,NaN,3:2005-cv-04374,True,3:2005-cv-04374_California Northern District C...,California Northern District Court,3:2005-cv-04374
50101,53512,3:05-cv-04374,35950.0,"Celerity, Inc. v. Ultra Clean Holding Inc. et al",UNITED STATES DISTRICT COURT NORTHERN DISTRICT...,Hon. Maxine M. Chesney,Magistrate Judge James Larson,35:271 Patent Infringement,Federal Question,NaN,...,NaN,NaN,2005-10-27,2007-11-30,NaN,3:2005-cv-04374,True,3:2005-cv-04374_California Northern District C...,California Northern District Court,3:2005-cv-04374


In [93]:
def evaluate_match(df1, df2, key):
    set2 = set(df2[key])

    df1['is_matched'] = df1[key].isin(set2)

    total = len(df1)
    matched = df1['is_matched'].sum()
    unmatched = total - matched

    print(f"Total: {total}")
    print(f"Matched: {matched}")
    print(f"Unmatched: {unmatched}")
    print(f"Match rate: {matched / total:.2%}")

    return df1

In [95]:
def evaluate_match(df1, df2, key1, key2):
    set2 = set(df2[key2])

    df1['is_matched'] = df1[key1].isin(set2)

    total = len(df1)
    matched = df1['is_matched'].sum()
    unmatched = total - matched

    print(f"Total: {total}")
    print(f"Matched: {matched}")
    print(f"Unmatched: {unmatched}")
    print(f"Match rate: {matched / total:.2%}")

    return df1

In [96]:
# How many cases matched into pacer
cases = evaluate_match(cases, pacer, 'case_number_normalized', 'p_case_number_norm')

# How many pacer records matched into cases
pacer = evaluate_match(pacer, cases, 'p_case_number_norm', 'case_number_norm')

Total: 74629
Matched: 74629
Unmatched: 0
Match rate: 100.00%
Total: 74852
Matched: 74644
Unmatched: 208
Match rate: 99.72%


In [98]:
# How many cases matched into pacer
cases = evaluate_match(cases, pacer, 'case_number_normalized', 'p_case_number_norm')

# How many pacer records matched into cases
pacer = evaluate_match(pacer, cases, 'p_case_number_norm', 'case_number_norm')

Total: 74629
Matched: 74629
Unmatched: 0
Match rate: 100.00%
Total: 74852
Matched: 74644
Unmatched: 208
Match rate: 99.72%


In [99]:
cases['identifier'] = cases['case_number_normalized'] + '_' + cases['court_name_normalized']
pacer['identifier'] = pacer['p_case_number_norm'] + '_' + pacer['court_name']
cases = evaluate_match(cases, pacer, 'identifier', 'identifier')
pacer = evaluate_match(pacer, cases, 'identifier', 'identifier')

Total: 74629
Matched: 74625
Unmatched: 4
Match rate: 99.99%
Total: 74852
Matched: 74618
Unmatched: 234
Match rate: 99.69%


In [100]:
cases[cases['is_matched'] == False][['identifier', 'case_number_normalized', 'court_name_normalized', 'case_name']]

,identifier,case_number_normalized,court_name_normalized,case_name
1885,1:2012-cv-05363_New Jersey District Court,1:2012-cv-05363,New Jersey District Court,"CLIO USA, INC. v. THE PROCTER & GAMBLE COMPANY"
12825,2:2011-cv-00494_Maryland District Court,2:2011-cv-00494,Maryland District Court,"Webvention LLC v. GNC Holdings, Inc. et al"
66912,6:2008-cv-00494_Texas Northern District Court,6:2008-cv-00494,Texas Northern District Court,"Saxon Innovations, LLC v. Nokia Corp. et al"
70570,7:1994-cv-00063_Texas Western District Court,7:1994-cv-00063,Texas Western District Court,"Whiteman Industries v. Stone Construction, et al"


In [ ]:
# Get the pacer identifiers for these case numbers
case_nums = ['1:2012-cv-05363', '2:2011-cv-00494', '6:2008-cv-00494', '7:1994-cv-00063']

pacer_identifiers = pacer[pacer['p_case_number_norm'].isin(case_nums)][['p_case_number_norm', 'identifier']]
print(pacer_identifiers)

In [101]:
# Build mapping from case_number -> pacer identifier
pacer_id_mapping = pacer.set_index('p_case_number_norm')['identifier'].to_dict()

# Overwrite only the 4 unmatched rows
unmatched_idx = cases[cases['is_matched'] == False].index
cases.loc[unmatched_idx, 'identifier'] = cases.loc[unmatched_idx, 'case_number_normalized'].map(pacer_id_mapping)

In [102]:
cases = evaluate_match(cases, pacer, 'identifier', 'identifier')
pacer = evaluate_match(pacer, cases, 'identifier', 'identifier')

Total: 74629
Matched: 74629
Unmatched: 0
Match rate: 100.00%
Total: 74852
Matched: 74619
Unmatched: 233
Match rate: 99.69%


In [103]:
print("=== Cases duplicates ===")
print(cases[cases['identifier'].duplicated(keep=False)].sort_values('identifier').to_string())

print("\n=== Pacer duplicates ===")
print(pacer[pacer['identifier'].duplicated(keep=False)].sort_values('identifier').to_string())


=== Cases duplicates ===
       case_row_id            case_number  pacer_id                                                                            case_name                                                    court_name                    assigned_to                         referred_to                     case_cause jurisdictional_basis     demand jury_demand lead_case                                                                                 related_case settlement  date_filed date_closed date_last_filed case_number_normalized  is_matched                                          identifier               court_name_normalized case_number_norm
6382         12558          1:05-cv-00590   46693.0                          Jackson Products, Inc. v. Fibre-Metal Products Company, The     UNITED STATES DISTRICT COURT WESTERN DISTRICT OF MICHIGAN         Judge Robert J. Jonker                                 NaN     35:271 Patent Infringement     Federal Question        NaN   Defendant

In [86]:
pacer_clean = pacer[~pacer['case_number'].str.contains('font', case=False, na=False)]
print(f"Pacer: {len(pacer)} -> {len(pacer_clean)}")

Pacer: 74953 -> 74853


In [87]:
print("\n=== Pacer duplicates ===")
print(pacer[pacer['identifier'].duplicated(keep=False)].sort_values('identifier').to_string())


=== Pacer duplicates ===
                                                                                                                                                          case_name court_code                       court_name date_closed                                                                                           case_number  pacer_id  date_filed                                                                          pacer_case_number_normalized  is_matched                                                                                                                          identifier                                                                                    p_case_number_norm
72466  Eckel Manufacturing Co., Inc. v. Superior Manufacturing & Hydraulics, Inc. et al <B><font color=red>DO NOT DOCKET. CASE HAS BEEN TRANSFERRED OUT.</font></B>       txsd    Texas Southern District Court  04/30/2008                         <b><font color="red">DO NOT DOCKET. CA